---
# **Cuaderno 3:**  Extraer mutaciones desde Clinvar
---
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: May 2026

**Aims:**


In [11]:
#Dependences
import pandas as pd
import numpy as np
import requests
import difflib

## Processing clinvar files

In [123]:
list_genes = ['BMPR1B', 'ACVR2A', 'BMPR1A', 'ACVR1B', 'TGFBR1', 'ACVRL1','TGFBR2', 'ACVR1', 'ACVR2B', 'BMPR2', 'AMHR2', 'ACVR1C']

dir_path = "../data/Genes_Clinvar"

files_df = []

for gen in list_genes:
    path_file = f"{dir_path}/{gen}.txt"
    df = pd.read_csv(path_file, sep="\t")
    df['gene_name'] = gen
    files_df.append(df)

    df_concat = pd.concat(files_df, ignore_index=True)
# Remove unnecessary columns
columns_drop = ['Name', 'Gene(s)', 'Accession','GRCh37Chromosome', 'GRCh37Location',  'Germline date last evaluated',
       'Germline review status', 'Somatic clinical impact',
       'Somatic clinical impact date last evaluated',
       'Somatic clinical impact review status', 'Oncogenicity classification',
       'Oncogenicity date last evaluated', 'Oncogenicity review status',
       'Unnamed: 24']

df_clinvar = df_concat.drop(columns=columns_drop)
df_clinvar['Protein change'] = df_clinvar['Protein change'].astype(str).fillna('')
df_clinvar['Molecular consequence'] = df_clinvar['Molecular consequence'].astype(str).fillna('')
df_clinvar['Condition(s)'] = df_clinvar['Condition(s)' ].astype(str).fillna('')

# Convert to a list using the delimiters “|” and “,” 
df_clinvar['Protein change'] = df_clinvar['Protein change'].str.split(', ')
df_clinvar['Molecular consequence'] = df_clinvar['Molecular consequence'].str.split('|')
df_clinvar['Condition(s)'] = df_clinvar['Condition(s)'].str.split('|')

# Processing data 
max_len = df_clinvar[['Protein change', 'Molecular consequence', 'Condition(s)']].map(len).max(axis=1)
# sort the list by size
for col in ['Protein change', 'Molecular consequence', 'Condition(s)']:
    df_clinvar[col] = df_clinvar.apply(
        lambda row: row[col] + [''] * (max_len[row.name] - len(row[col])), 
        axis=1
    )
# Explode
df_clinvar = df_clinvar.explode(['Protein change', 'Molecular consequence', 'Condition(s)'])
df_clinvar = df_clinvar.rename(columns={'Condition(s)': 'Condition'}) #rename column

# Clean columns Protein change y Molecular Consequence
df_clinvar['Protein change'] = df_clinvar['Protein change'].astype(str).str.replace(r"[\[\]'’]", "", regex=True)
df_clinvar['Molecular consequence'] = df_clinvar['Molecular consequence'].astype(str).str.replace(r"[\[\]'’]", "", regex=True)

# Convert rows to Nan, because they were formatted as [nan]
df_clinvar['Protein change'] = df_clinvar['Protein change'].replace('nan', np.nan)
df_clinvar['Molecular consequence'] = df_clinvar['Molecular consequence'].replace('nan', np.nan)

enfermedades_por_mutacion.columns = ['ID_Mutacion', 'Disease'] # rename columns

# dictionary of diseases, for mapping by variant
diccionario_enfermedades = df_clinvar.groupby('VariationID')['Condition'].unique().apply(
    lambda x: [enf for enf in x if pd.notna(enf) and enf != '']
)
df_clinvar['Disease'] = df_clinvar['VariationID'].map(diccionario_enfermedades)

#drop duplicates
df_clinvar = df_clinvar.drop_duplicates(subset=['VariationID'])
#drop nan
df_clinvar = df_clinvar.dropna(subset=["Protein change"])

#dataframe
df_clinvar['Disease'] = df_clinvar['Disease'].apply(lambda x: '; '.join(x) if isinstance(x, list) else x)
df_clinvar['Disease'] = df_clinvar['Disease'].astype(str).str.replace(r"[\[\]'’]", "", regex=True).fillna('')
df_clinvar['Disease'] = df_clinvar['Disease'].str.split('; ')
df_clinvar = df_clinvar.explode('Disease')
df_clinvar['Disease'] = df_clinvar['Disease'].str.strip()

#Reorder columns
df_clinvar = df_clinvar[['gene_name','Protein change', 'Disease', 'Variant type','Molecular consequence', 'Germline classification','GRCh38Chromosome', 'GRCh38Location',
                         'VariationID', 'AlleleID(s)', 'dbSNP ID' ,'Canonical SPDI']]

exclude=["Inborn genetic diseases", "not provided", "AMHR2-related disorder", "Hereditary cancer-predisposing syndrome", "See cases", "TGFBR2-related disorder", "not specified", ]

df_clinvar = df_clinvar.loc[~df_clinvar["Disease"].isin(exclude)]

df_clinvar



,gene_name,Protein change,Disease,Variant type,Molecular consequence,Germline classification,GRCh38Chromosome,GRCh38Location,VariationID,AlleleID(s),dbSNP ID,Canonical SPDI
6,BMPR1B,R31C,Acromesomelic dysplasia 3,single nucleotide variant,missense variant,Pathogenic,4.0,95104515,217256,213899,rs745854387,NC_000004.12:95104514:C:T
7,BMPR1B,C53R,Acromesomelic dysplasia 3,single nucleotide variant,missense variant,Pathogenic,4.0,95114733,217254,213897,rs863225041,NC_000004.12:95114732:T:C
9,BMPR1B,G151fs,Acromesomelic dysplasia 3,Deletion,frameshift variant,Pathogenic,4.0,95123819 - 95123826,6557,21596,rs863223287,NC_000004.12:95123818:ATGGACCTAT:AT
10,BMPR1B,C165fs,Type A2 brachydactyly,Microsatellite,frameshift variant,Pathogenic,4.0,95123858 - 95123861,2926869,3090338,rs1560671891,NC_000004.12:95123857:CTGTCTGT:CTGT
10,BMPR1B,C165fs,Acromesomelic dysplasia 3,Microsatellite,frameshift variant,Pathogenic,4.0,95123858 - 95123861,2926869,3090338,rs1560671891,NC_000004.12:95123857:CTGTCTGT:CTGT
...,...,...,...,...,...,...,...,...,...,...,...,...
2019,AMHR2,R407*,Persistent Mullerian duct syndrome,single nucleotide variant,nonsense,Pathogenic,12.0,53429909,3779441,3895423,NaN,NC_000012.12:53429908:C:T
2021,AMHR2,P446S,Persistent Mullerian duct syndrome,single nucleotide variant,missense variant,Pathogenic,12.0,53430197,1707557,1705861,rs2498922382,NC_000012.12:53430196:C:T
2022,AMHR2,R463C,Persistent Mullerian duct syndrome,single nucleotide variant,synonymous variant,Pathogenic/Likely pathogenic,12.0,53430244,372896,360131,rs374247138,NC_000012.12:53430243:C:T
2023,AMHR2,R471H,Persistent Mullerian duct syndrome,single nucleotide variant,missense variant,Pathogenic/Likely pathogenic,12.0,53430269,2884050,3043352,rs767904039,NC_000012.12:53430268:G:A


## Extraer orphacode desde uniprot

In [92]:

uniprot_ids = [
    'O00238', 'P27037', 'P36894', 'P35790', 'P36897', 'P37023',
    'P37173', 'Q04771', 'Q13705', 'Q13873', 'Q16671', 'Q8NER5'
]

resultados = []

for uniprot_id in uniprot_ids:
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error al obtener {uniprot_id}: {response.status_code}")
        continue
    
    json_data = response.json()
    
    # Gene name
    gen_name = (json_data.get("genes", [{}])[0].get("geneName", {}) or {}).get("value", uniprot_id)
    
    cross_references = json_data.get("uniProtKBCrossReferences", [])
    
    for cross in cross_references:
        if cross.get("database") == "Orphanet":
            orphan_code = cross.get("id")
            props = {p.get("key"): p.get("value") for p in cross.get("properties", [])}
            resultados.append({
                "Hugo_Symbol": gen_name,      
                "UniProt_ID": uniprot_id,     
                "Database": "Orphanet",
                "OrphanetCode": orphan_code,
                "DiseaseName": props.get("Disease", "N/A")
            })

# Dataframe
if resultados:
    df_orphan = pd.DataFrame(resultados)
else:
    print("\nNinguno de los genes analizados devolvió conexiones directas con Orphanet.")
df_orphan.head(2)

,Hugo_Symbol,UniProt_ID,Database,OrphanetCode,DiseaseName
0,BMPR1B,O00238,Orphanet,2098,"Acromesomelic dysplasia, Grebe type"
1,BMPR1B,O00238,Orphanet,93388,Brachydactyly type A1


## Mapear RD 

In [120]:
from difflib import get_close_matches

"""Next, manual curing must be performed"""

enf_raras = df_orphan["DiseaseName"].tolist()
def buscar_match(enfermedad):
    match = get_close_matches(enfermedad, enf_raras, n=3, cutoff=0.47)
    return match[0] if match else "No coincidences"

df_clinvar["Orphanet_Match"] = df_clinvar["Disease"].apply(buscar_match)

# results
print(df_clinvar[["Disease", "Orphanet_Match"]].dropna())

                                 Disease                       Orphanet_Match
6              Acromesomelic dysplasia 3  Acromesomelic dysplasia, Grebe type
7              Acromesomelic dysplasia 3  Acromesomelic dysplasia, Grebe type
9              Acromesomelic dysplasia 3  Acromesomelic dysplasia, Grebe type
10                 Type A2 brachydactyly                 Brachydactyly type C
10             Acromesomelic dysplasia 3  Acromesomelic dysplasia, Grebe type
...                                  ...                                  ...
2019  Persistent Mullerian duct syndrome  Persistent Muellerian duct syndrome
2021  Persistent Mullerian duct syndrome  Persistent Muellerian duct syndrome
2022  Persistent Mullerian duct syndrome  Persistent Muellerian duct syndrome
2023  Persistent Mullerian duct syndrome  Persistent Muellerian duct syndrome
2025  Persistent Mullerian duct syndrome  Persistent Muellerian duct syndrome

[1531 rows x 2 columns]


In [121]:
# save df
df_clinvar.to_csv("clinvar_orphanet.csv", sep="\t")